# EV Intelligence Analytics - Exploratory Data Analysis

## Overview
This notebook performs comprehensive exploratory data analysis on the EV market dataset to uncover insights, patterns, and trends in electric vehicle adoption across different states and manufacturers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [ ]:
# Load processed data
ev_sales = pd.read_csv('../data/processed/ev_sales_clean.csv')
charging_stations = pd.read_csv('../data/processed/charging_stations_clean.csv')
market_metrics = pd.read_csv('../data/processed/market_metrics_clean.csv')
monthly_sales = pd.read_csv('../data/processed/monthly_sales.csv')
manufacturer_performance = pd.read_csv('../data/processed/manufacturer_performance.csv')
merged_data = pd.read_csv('../data/processed/merged_analytics_data.csv')

print("Datasets loaded successfully!")
print(f"EV Sales: {ev_sales.shape}")
print(f"Charging Stations: {charging_stations.shape}")
print(f"Market Metrics: {market_metrics.shape}")
print(f"Monthly Sales: {monthly_sales.shape}")
print(f"Manufacturer Performance: {manufacturer_performance.shape}")
print(f"Merged Analytics Data: {merged_data.shape}")

## 1. Market Overview Analysis

In [ ]:
# Overall market statistics
total_sales = ev_sales['sales_amount'].sum()
total_revenue = ev_sales['revenue'].sum()
total_states = ev_sales['state'].nunique()
total_manufacturers = ev_sales['manufacturer'].nunique()
avg_penetration = market_metrics['ev_penetration_rate'].mean()

print("=== MARKET OVERVIEW ===")
print(f"Total EV Sales: {total_sales:,} units")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Average Vehicle Price: ${total_revenue/total_sales:,.2f}")
print(f"States Covered: {total_states}")
print(f"Manufacturers: {total_manufacturers}")
print(f"Average Penetration Rate: {avg_penetration:.2%}")

# Create summary dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sales by State', 'Revenue by State', 'Market Penetration', 'Manufacturer Distribution'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "pie"}]]
)

# Sales by state
state_sales = ev_sales.groupby('state')['sales_amount'].sum().sort_values(ascending=False).head(10)
fig.add_trace(
    go.Bar(x=state_sales.index, y=state_sales.values, name='Sales', marker_color='#1f77b4'),
    row=1, col=1
)

# Revenue by state
state_revenue = ev_sales.groupby('state')['revenue'].sum().sort_values(ascending=False).head(10)
fig.add_trace(
    go.Bar(x=state_revenue.index, y=state_revenue.values, name='Revenue', marker_color='#ff7f0e'),
    row=1, col=2
)

# Market penetration
state_penetration = market_metrics.groupby('state')['ev_penetration_rate'].mean().sort_values(ascending=False).head(10)
fig.add_trace(
    go.Bar(x=state_penetration.index, y=state_penetration.values, name='Penetration', marker_color='#2ca02c'),
    row=2, col=1
)

# Manufacturer distribution
manufacturer_sales = ev_sales.groupby('manufacturer')['sales_amount'].sum().sort_values(ascending=False)
fig.add_trace(
    go.Pie(labels=manufacturer_sales.index, values=manufacturer_sales.values, name="Market Share"),
    row=2, col=2
)

fig.update_layout(height=800, showlegend=False, title_text="EV Market Overview Dashboard")
fig.show()

## 2. Temporal Analysis

In [ ]:
# Convert date columns
ev_sales['date'] = pd.to_datetime(ev_sales['date'])
monthly_sales['month_year'] = pd.to_datetime(monthly_sales['month_year'])

# Monthly trends
monthly_trends = monthly_sales.groupby('month_year').agg({
    'sales_amount': 'sum',
    'revenue': 'sum',
    'sales_growth_pct': 'mean'
}).reset_index()

# Create time series plots
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Monthly Sales Volume', 'Monthly Revenue', 'Sales Growth Rate'),
    vertical_spacing=0.08
)

# Sales volume
fig.add_trace(
    go.Scatter(x=monthly_trends['month_year'], y=monthly_trends['sales_amount'], 
               mode='lines+markers', name='Sales Volume', line=dict(color='#1f77b4', width=3)),
    row=1, col=1
)

# Revenue
fig.add_trace(
    go.Scatter(x=monthly_trends['month_year'], y=monthly_trends['revenue'], 
               mode='lines+markers', name='Revenue', line=dict(color='#ff7f0e', width=3)),
    row=2, col=1
)

# Growth rate
fig.add_trace(
    go.Scatter(x=monthly_trends['month_year'], y=monthly_trends['sales_growth_pct'], 
               mode='lines+markers', name='Growth Rate', line=dict(color='#2ca02c', width=3)),
    row=3, col=1
)

fig.update_layout(height=900, title_text="EV Market Temporal Analysis")
fig.show()

# Growth statistics
print("=== GROWTH ANALYSIS ===")
print(f"Average Monthly Growth: {monthly_trends['sales_growth_pct'].mean():.2f}%")
print(f"Highest Growth Month: {monthly_trends.loc[monthly_trends['sales_growth_pct'].idxmax(), 'month_year'].strftime('%Y-%m')}")
print(f"Peak Sales Month: {monthly_trends.loc[monthly_trends['sales_amount'].idxmax(), 'month_year'].strftime('%Y-%m')}")

## 3. State-wise Deep Dive

In [ ]:
# State performance analysis
state_analysis = merged_data.groupby('state').agg({
    'sales_amount': 'sum',
    'revenue': 'sum',
    'ev_penetration_rate': 'mean',
    'gdp_per_capita': 'mean',
    'total_stations': 'mean',
    'stations_per_1000_ev': 'mean',
    'ev_density': 'mean'
}).reset_index()

# Create state comparison dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sales vs Penetration', 'GDP per Capita vs EV Density', 
                   'Charging Infrastructure', 'State Performance Heatmap'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "bar"}, {"type": "choropleth"}]]
)

# Sales vs Penetration
fig.add_trace(
    go.Scatter(x=state_analysis['ev_penetration_rate'], y=state_analysis['sales_amount'],
               mode='markers+text', text=state_analysis['state'], textposition="top center",
               marker=dict(size=state_analysis['revenue']/1000000, color=state_analysis['sales_amount'],
                          colorscale='Viridis', showscale=True), name='States'),
    row=1, col=1
)

# GDP per Capita vs EV Density
fig.add_trace(
    go.Scatter(x=state_analysis['gdp_per_capita'], y=state_analysis['ev_density'],
               mode='markers+text', text=state_analysis['state'], textposition="top center",
               marker=dict(size=12, color='#ff7f0e'), name='States'),
    row=1, col=2
)

# Charging infrastructure
top_states = state_analysis.nlargest(10, 'sales_amount')
fig.add_trace(
    go.Bar(x=top_states['state'], y=top_states['total_stations'], name='Charging Stations', marker_color='#2ca02c'),
    row=2, col=1
)

# State performance heatmap
performance_matrix = state_analysis.set_index('state')[['sales_amount', 'revenue', 'ev_penetration_rate', 'ev_density']]
fig.add_trace(
    go.Heatmap(z=performance_matrix.values, x=performance_matrix.columns, y=performance_matrix.index,
               colorscale='RdYlBu', name='Performance'),
    row=2, col=2
)

fig.update_layout(height=900, title_text="State-wise EV Market Analysis")
fig.show()

# Top performing states
print("=== TOP PERFORMING STATES ===")
top_states_analysis = state_analysis.nlargest(5, 'sales_amount')[['state', 'sales_amount', 'revenue', 'ev_penetration_rate']]
print(top_states_analysis.to_string(index=False))

## 4. Manufacturer Competitive Analysis

In [ ]:
# Manufacturer performance analysis
manufacturer_analysis = ev_sales.groupby('manufacturer').agg({
    'sales_amount': 'sum',
    'revenue': 'sum',
    'price_avg': 'mean',
    'battery_capacity': 'mean',
    'market_share': 'mean'
}).reset_index()

# Calculate market share percentages
manufacturer_analysis['market_share_pct'] = (manufacturer_analysis['sales_amount'] / manufacturer_analysis['sales_amount'].sum()) * 100
manufacturer_analysis = manufacturer_analysis.sort_values('sales_amount', ascending=False)

# Create manufacturer dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Market Share Distribution', 'Average Price vs Sales', 
                   'Battery Capacity Comparison', 'State Presence Analysis'),
    specs=[[{"type": "pie"}, {"type": "scatter"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# Market share pie chart
fig.add_trace(
    go.Pie(labels=manufacturer_analysis['manufacturer'], values=manufacturer_analysis['sales_amount'],
           name="Market Share"),
    row=1, col=1
)

# Price vs Sales scatter
fig.add_trace(
    go.Scatter(x=manufacturer_analysis['price_avg'], y=manufacturer_analysis['sales_amount'],
               mode='markers+text', text=manufacturer_analysis['manufacturer'], textposition="top center",
               marker=dict(size=manufacturer_analysis['market_share_pct']*2, color='#ff7f0e'),
               name='Manufacturers'),
    row=1, col=2
)

# Battery capacity comparison
fig.add_trace(
    go.Bar(x=manufacturer_analysis['manufacturer'], y=manufacturer_analysis['battery_capacity'],
           name='Avg Battery Capacity', marker_color='#2ca02c'),
    row=2, col=1
)

# State presence
state_presence = ev_sales.groupby('manufacturer')['state'].nunique().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=state_presence.index, y=state_presence.values, name='States Present', marker_color='#d62728'),
    row=2, col=2
)

fig.update_layout(height=800, title_text="Manufacturer Competitive Analysis")
fig.show()

# Manufacturer insights
print("=== MANUFACTURER INSIGHTS ===")
print(f"Market Leader: {manufacturer_analysis.iloc[0]['manufacturer']} ({manufacturer_analysis.iloc[0]['market_share_pct']:.1f}% market share)")
print(f"Premium Brand: {manufacturer_analysis.loc[manufacturer_analysis['price_avg'].idxmax(), 'manufacturer']} (${manufacturer_analysis['price_avg'].max():,.0f} avg price)")
print(f"Best Battery Tech: {manufacturer_analysis.loc[manufacturer_analysis['battery_capacity'].idxmax(), 'manufacturer']} ({manufacturer_analysis['battery_capacity'].max():.0f} kWh)")
print(f"Widest Reach: {state_presence.index[0]} ({state_presence.iloc[0]} states)")

## 5. Vehicle Category Analysis

In [ ]:
# Vehicle category analysis
category_analysis = ev_sales.groupby(['vehicle_category', 'vehicle_type']).agg({
    'sales_amount': 'sum',
    'revenue': 'sum',
    'price_avg': 'mean',
    'battery_capacity': 'mean',
    'charging_time': 'mean'
}).reset_index()

# Create vehicle category dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sales by Category', 'Average Price by Type', 
                   'Battery Capacity vs Charging Time', 'Category Market Share'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "pie"}]]
)

# Sales by category
category_sales = category_analysis.groupby('vehicle_category')['sales_amount'].sum().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=category_sales.index, y=category_sales.values, name='Sales', marker_color='#1f77b4'),
    row=1, col=1
)

# Average price by type
type_price = category_analysis.groupby('vehicle_type')['price_avg'].mean().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=type_price.index, y=type_price.values, name='Avg Price', marker_color='#ff7f0e'),
    row=1, col=2
)

# Battery vs charging time
fig.add_trace(
    go.Scatter(x=category_analysis['battery_capacity'], y=category_analysis['charging_time'],
               mode='markers+text', text=category_analysis['vehicle_type'], textposition="top center",
               marker=dict(size=category_analysis['sales_amount']/100, color='#2ca02c'),
               name='Vehicle Types'),
    row=2, col=1
)

# Category market share
fig.add_trace(
    go.Pie(labels=category_sales.index, values=category_sales.values, name="Category Share"),
    row=2, col=2
)

fig.update_layout(height=800, title_text="Vehicle Category Analysis")
fig.show()

# Category insights
print("=== VEHICLE CATEGORY INSIGHTS ===")
print(f"Most Popular Category: {category_sales.index[0]} ({category_sales.iloc[0]:,} units)")
print(f"Highest Priced Type: {type_price.index[0]} (${type_price.iloc[0]:,.0f})")
print(f"Best Battery/Capacity: {category_analysis.loc[category_analysis['battery_capacity'].idxmax(), 'vehicle_type']}")

## 6. Correlation Analysis

In [ ]:
# Create correlation matrix
correlation_data = merged_data[[
    'sales_amount', 'revenue', 'market_share', 'battery_capacity', 'price_avg',
    'ev_penetration_rate', 'gdp_per_capita', 'gasoline_price', 'total_stations',
    'stations_per_1000_ev', 'ev_density'
]].corr()

# Create correlation heatmap
fig = go.Figure(data=go.Heatmap(
    z=correlation_data.values,
    x=correlation_data.columns,
    y=correlation_data.columns,
    colorscale='RdBu',
    zmid=0,
    text=correlation_data.round(2).values,
    texttemplate="%{text}",
    textfont={"size": 10},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='Feature Correlation Matrix',
    width=900,
    height=900
)

fig.show()

# Key correlations with sales
sales_correlations = correlation_data['sales_amount'].sort_values(ascending=False)
print("=== KEY CORRELATIONS WITH SALES ===")
for feature, corr in sales_correlations.items():
    if feature != 'sales_amount' and abs(corr) > 0.3:
        print(f"{feature}: {corr:.3f}")

## 7. Key Insights Summary

In [ ]:
# Generate key insights
insights = {
    'market_overview': {
        'total_sales': int(ev_sales['sales_amount'].sum()),
        'total_revenue': float(ev_sales['revenue'].sum()),
        'avg_penetration': float(market_metrics['ev_penetration_rate'].mean()),
        'top_state': state_analysis.iloc[0]['state'],
        'top_manufacturer': manufacturer_analysis.iloc[0]['manufacturer']
    },
    'growth_trends': {
        'avg_monthly_growth': float(monthly_trends['sales_growth_pct'].mean()),
        'peak_sales_month': monthly_trends.loc[monthly_trends['sales_amount'].idxmax(), 'month_year'].strftime('%Y-%m'),
        'highest_growth_state': state_analysis.loc[state_analysis['ev_penetration_rate'].idxmax(), 'state']
    },
    'infrastructure': {
        'total_charging_stations': int(charging_stations['total_stations'].sum()),
        'avg_utilization': float(charging_stations['utilization_rate'].mean()),
        'best_infrastructure_state': state_analysis.loc[state_analysis['stations_per_1000_ev'].idxmax(), 'state']
    }
}

print("=== KEY BUSINESS INSIGHTS ===")
print(f"🚗 Market Size: {insights['market_overview']['total_sales']:,} EVs sold (${insights['market_overview']['total_revenue']:,.2f} revenue)")
print(f"📍 Market Leader: {insights['market_overview']['top_manufacturer']} dominates the market")
print(f"🏆 Top Market: {insights['market_overview']['top_state']} leads in sales volume")
print(f"📈 Growth Rate: {insights['growth_trends']['avg_monthly_growth']:.1f}% average monthly growth")
print(f"⚡ Infrastructure: {insights['infrastructure']['total_charging_stations']:,} charging stations nationwide")
print(f"🎯 Best Adoption: {insights['growth_trends']['highest_growth_state']} shows highest penetration")

# Save insights to file
import json
with open('../reports/eda_insights.json', 'w') as f:
    json.dump(insights, f, indent=2, default=str)

print("\n✅ EDA completed successfully! Insights saved to reports/eda_insights.json")